In [271]:
import pandas as pd
import numpy as np
from combat.pycombat import pycombat  # pip install combat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go  # For heatmaps

In [272]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# MAIN

In [273]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
# MEMO: Cannot compute epiSize with 'full values' (almost NO '0' values)
#epiSize = {}
#for epiSign in [x for x in X.columns if x not in ('coord')]:
#    epiSize[epiSign] = sum(X[epiSign] > 0)  # Catch 100% methyl

# Detect NaN or null values:
cols_with_na = {}
for a_col in X.columns:
    nb_na = sum(X[a_col].isna())
    if nb_na > 0:
        cols_with_na[a_col] = nb_na
assert cols_with_na, print("ERROR: Some cols have missing values:", cols_with_na)

TypeError: list indices must be integers or slices, not str

## Pre-processing

### Batch-effect correction using (py)combat

In [ ]:
# First we generate the list of batches:
dataset_251217 = ["HG002_combined","barcode04_combined"]
ref_sign = [ x for x in X.columns if x not in dataset_251217+['coord'] ]

batch = []
datasets = [ref_sign, dataset_251217]
for j in range(len(datasets)):
    batch.extend([j for _ in range(len(datasets[j]))])

# Then run (py)combat:
#X_corrected = pycombat(X, batch)

### Transpose + normalize

In [ ]:
USE_COMBAT = False
USE_NORMALIZED = False
EPISIGN_OF_INTEREST = ['HG002_combined','RMNS','Kleefstra','Kabuki','barcode04_combined']
# ONLY if 'full data from publi':
EPISIGN_OF_INTEREST += ['EPI_01', 'EPI_02', 'EPI_08','EPI_19'] # 4 kab samples

# Transpose (required):
X_t = X.T
if USE_COMBAT:
    X_t = X_corrected.T

# Remove 2nd row = size of epiSign then normalize
to_norm = X_t
if 'epiSize' in X_t.columns:
    to_norm = X_t.drop('epiSize', axis=1)
print(to_norm[to_norm.columns[0:3]].head())

# WARN: If norm, should be AFTER combat
to_PCA = to_norm
if USE_NORMALIZED:
    to_PCA = pd.DataFrame(StandardScaler().fit_transform(to_norm), columns=to_norm.columns, index=to_norm.index)

# Write corrected file:
for sample in ['barcode04_combined', 'HG002_combined']:
    #to_PCA.T[sample].to_csv(f"{sample}_corrected.tsv", sep="\t")
    print(f" > Wrote: {sample}_corrected.tsv")

## Heatmaps

In [ ]:
# Simple one
# MEMO: Have to drop coord, otherwise does not plot correctly (too many rows probably)
print(EPISIGN_OF_INTEREST)
fig = px.imshow(
    to_PCA.T[EPISIGN_OF_INTEREST].T.reset_index(drop=True),
    text_auto=True
)
fig.update_layout(
    yaxis=dict(tickfont_size=7)
)

In [ ]:
# MEMO: With index, plot broken with 'px.imshow'
names = to_PCA.index
coord = to_PCA.columns

print(to_PCA.info())

go.Figure(data=go.Heatmap(
    z = to_PCA.to_numpy(),
    x = coord,
    y = names
))

In [ ]:
# MEMO: Have to drop index, otherwise plot broken
# Width/height empirical bellow
# Correct param is 'tickfont' (and not 'textfont')
fig = px.imshow(
    to_PCA.to_numpy(),
    y = names,
    x = coord,
    width = 15000,
    height = 1000
)
fig.update_layout(
    autosize=False,
    xaxis=dict(tickfont=dict(size=5)),
    yaxis=dict(tickfont=dict(size=5))
)

## PCA

In [ ]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(to_PCA.to_numpy())

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [ ]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[EPISIGN_OF_INTEREST])

In [ ]:
# Plot PCA
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

## t-SNE

In [ ]:
# Run t-SNE:
# MEMOs:
# - In Joris' paper they use 'preplex=2'
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=2).fit_transform(to_PCA)

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_tsne = tsne_df.loc[EPISIGN_OF_INTEREST]
print(subset_tsne)

In [ ]:
# Plot t-SNE
color_selected = [x in EPISIGN_OF_INTEREST for x in tsne_df.index]
fig = px.scatter(
        tsne_df,
        x='compon0',
        y='compon1',
        hover_data=[tsne_df.index],
        color=color_selected
)
fig.show()